# 01 — Data inspection (step 2)

Reads the tracked inventories in `data/inventory/` (works without raw data) and, where the raw data
is present locally, shows original images with annotations. Reproduction commands: `data/README.md`.
Findings: `docs/validation_results.md`.

In [ ]:
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INV = ROOT / "data" / "inventory"
sources = json.loads((ROOT / "data" / "sources.json").read_text())
pd.DataFrame(
    {
        k: {"status": v.get("status"), "needs": ", ".join(v.get("needs", []))}
        for k, v in sources.items()
    }
).T

## SSDD — inventory, official split, duplicates

In [ ]:
ssdd_img = pd.read_csv(INV / "ssdd_images.csv", dtype={"image_id": str})
ssdd_box = pd.read_csv(INV / "ssdd_boxes.csv", dtype={"image_id": str})
rep = json.loads((INV / "ssdd_validation.json").read_text())["checks"]
print(pd.crosstab(ssdd_img.official_split, ssdd_img.scene_type, margins=True))
print(
    "boxes:",
    len(ssdd_box),
    "| exact-duplicate groups:",
    len(rep["exact_duplicate_groups"]),
    "| cross-split near-dup pairs:",
    rep["near_duplicate_cross_split_pairs"],
)
ssdd_box[["w", "h"]].describe().round(1)

In [ ]:
# Original image + COCO boxes (requires data/raw/ssdd). Duplicate pair across the official split:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, stem in zip(axes, ["001121", "001122"], strict=False):
    r = ssdd_img.set_index("image_id").loc[stem]
    p = ROOT / r.path
    if p.exists():
        ax.imshow(Image.open(p))
        for b in ssdd_box[ssdd_box.image_id == stem].itertuples():
            ax.add_patch(mpatches.Rectangle((b.x, b.y), b.w, b.h, fill=False, ec="lime"))
    ax.set_title(f"{stem} ({r.official_split})")
    ax.axis("off")
plt.show()

## xView3 — downloaded scenes and labels

In [ ]:
x3_scenes = pd.read_csv(INV / "xview3_scenes.csv")
x3_labels = pd.read_csv(ROOT / "data/processed/xview3_labels_downloaded.csv")  # local only
display(
    x3_scenes[
        [
            "scene_id",
            "esa_scene_name",
            "acquisition_start_utc",
            "crs",
            "width",
            "height",
            "n_labels",
            "n_vessel",
            "n_fishing",
        ]
    ]
)
pd.crosstab(x3_labels.scene_id, [x3_labels["class"]]).assign(
    max_geo_err_m=x3_labels.groupby("scene_id").pixel_to_geo_error_m.max().round(1)
)

In [ ]:
# A labelled chip read from the original VV raster (requires data/raw/xview3/scenes)
import rasterio
from rasterio.windows import Window

r = x3_labels[(x3_labels["class"] == "fishing") & (x3_labels.confidence == "HIGH")].iloc[0]
tif = ROOT / "data/raw/xview3/scenes" / r.scene_id / r.scene_id / "VV_dB.tif"
if tif.exists():
    with rasterio.open(tif) as src:
        chip = src.read(
            1, window=Window(r.detect_scene_column - 50, r.detect_scene_row - 50, 100, 100)
        )
    plt.imshow(chip.astype(float), cmap="gray")
    plt.plot(50, 50, "r+")
    plt.title(f"{r.scene_id} fishing vessel, L={r.vessel_length_m}")
    plt.show()

## GFW detection tiles and SAR detections (South China Sea)

In [ ]:
tiles = pd.read_csv(INV / "gfw_detection_tiles.csv", parse_dates=["acquisition_start_utc"])
print(
    tiles.presence.value_counts().to_dict(),
    "| scenes:",
    tiles.scene_id.nunique(),
    "| in SCS box:",
    tiles.in_scs_box.sum(),
)
scs = pd.read_csv(INV / "gfw_sar_scs_scenes.csv", parse_dates=["timestamp"])
print("SCS scenes:", len(scs))
scs.groupby(scs.timestamp.dt.year)[
    ["n", "unmatched", "matched_fishing", "matched_nonfishing", "matched_unknown"]
].sum()

**Reminder**: GFW SAR detections are GFW model predictions joined to AIS by GFW — not ground truth
and not raw AIS. "unmatched" ≠ "dark vessel" (AIS reception in the SCS was poor before ~2022).